In [ ]:
# STEP 0 PERSONAL PARAMETERS — this MUST be the first cell.
# Replace ONLY the value below with the last four digits of your SJSU student ID.
SID4_TEXT = "2738"

from pathlib import Path
import json, re, random
import numpy as np
import torch

if not re.fullmatch(r"\d{4}", SID4_TEXT):
    raise ValueError("Edit SID4_TEXT to exactly 4 digits.")

SID4 = int(SID4_TEXT)
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def resolve_assignment_dir():
    cwd = Path.cwd().resolve()

    # Local/GPU-lab checkout.
    for p in [cwd, *cwd.parents]:
        if p.name == "GPU_Assignment1" or (p / "README_FIRST.md").exists():
            return p

    # Google Colab / Google Drive.
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive")
        matches = list(root.rglob("GPU_Assignment1"))
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise RuntimeError(
                "Multiple GPU_Assignment1 folders found: "
                + ", ".join(str(x) for x in matches)
            )
    except ImportError:
        pass

    raise FileNotFoundError(
        "Could not locate GPU_Assignment1. Run from the assignment folder "
        "or mount Google Drive."
    )

ASSIGNMENT_DIR = resolve_assignment_dir()
ARTIFACT_DIR = ASSIGNMENT_DIR / "hw2_5_artifacts"
FIG_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PARAMS = {
    "SID4": SID4,
    "SEED": SEED,
    "SLICE": SLICE,
    "HP_ID": HP_ID,
    "CLS_A": CLS_A,
    "CLS_B": CLS_B,
}

PARAM_FILE = ARTIFACT_DIR / "student_params.json"
PARAM_FILE.write_text(json.dumps(PARAMS, indent=2))

print(f"SID4={SID4:04d} | SEED={SEED} | SLICE={SLICE} | HP_ID={HP_ID} | CLS_A={CLS_A} | CLS_B={CLS_B}")
print("Assignment:", ASSIGNMENT_DIR)
print("Artifacts:", ARTIFACT_DIR)


# HW2.5 — GPU Assignment I
## Notebook 00 — Step 0, onboarding, and provenance

Run this notebook **first** on the reserved RTX 4090 or RTX 5090 workstation.

This notebook implements the standing Step 0 requirements relevant to HW2.5:
- states `SID4`, `SEED`, `SLICE`, `HP_ID`, `CLS_A`, and `CLS_B` at the top;
- seeds Python/NumPy/PyTorch;
- records the environment and complete `nvidia-smi -q`;
- creates the shared `RUN_LOG.txt`;
- creates a reservation/GPU-hour record template;
- stores the card's NVIDIA-documented specs and dense theoretical peaks used later.

**NVIDIA sources used by the notebook**

- RTX 4090 / Ada architecture whitepaper:  
  https://images.nvidia.com/aem-dam/Solutions/geforce/ada/nvidia-ada-gpu-architecture.pdf
- RTX 5090 / Blackwell architecture whitepaper:  
  https://images.nvidia.com/aem-dam/Solutions/geforce/blackwell/nvidia-rtx-blackwell-gpu-architecture.pdf

The dense (non-sparse) Tensor TFLOPS values in the configuration use the **left-hand value** in NVIDIA's `dense / sparse` entries, and FP16/BF16 use the FP32-accumulate figure so the denominator matches the benchmark configuration.

In [ ]:
import os, sys, time, json, subprocess, platform, traceback
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch

# ARTIFACT_DIR / FIG_DIR were created by the first cell and are persistent.
FIG_DIR.mkdir(parents=True, exist_ok=True)
RUN_LOG = ARTIFACT_DIR / "RUN_LOG.txt"

# Change only if the reserved physical GPU is not nvidia-smi index 0.
NVIDIA_SMI_INDEX = 0

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def log(message=""):
    line = str(message)
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as f:
        f.write(f"[{utc_now()}] {line}\n")

def run_cmd(cmd, check=False):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {cmd}\nSTDOUT:\n{p.stdout}\nSTDERR:\n{p.stderr}"
        )
    return p

log("=" * 88)
log("HW2.5 Notebook 00 started")
log(f"Python: {sys.version.replace(chr(10), ' ')}")
log(f"Platform: {platform.platform()}")
log(f"PyTorch: {torch.__version__}")
log(f"PyTorch CUDA build: {torch.version.cuda}")
log(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Use the required RTX 4090/5090 workstation.")

log(f"Torch visible GPU 0: {torch.cuda.get_device_name(0)}")


## Part A.2 — capture `nvidia-smi -q` and provenance

The assignment requires the **complete** `nvidia-smi -q` output. The next cell saves it verbatim with a timestamp. It also stores a structured summary used by later notebooks.

In [ ]:
# Capture basic nvidia-smi output (contains the driver-supported CUDA version in the header).
basic = run_cmd(["nvidia-smi"], check=True)
(ARTIFACT_DIR / "nvidia_smi_basic.txt").write_text(basic.stdout)

m = re.search(r"CUDA Version:\s*([0-9.]+)", basic.stdout)
driver_cuda_version = m.group(1) if m else "not parsed"

# Complete -q output required by Part A.
q = run_cmd(["nvidia-smi", "-i", str(NVIDIA_SMI_INDEX), "-q"], check=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
q_path = ARTIFACT_DIR / f"nvidia_smi_q_gpu{NVIDIA_SMI_INDEX}_{stamp}.txt"
q_path.write_text(q.stdout)

query_fields = [
    "index", "name", "uuid", "driver_version", "memory.total", "power.limit"
]
query = run_cmd([
    "nvidia-smi", "-i", str(NVIDIA_SMI_INDEX),
    "--query-gpu=" + ",".join(query_fields),
    "--format=csv,noheader,nounits"
], check=True)

vals = [x.strip() for x in query.stdout.strip().split(",")]
if len(vals) != len(query_fields):
    raise RuntimeError(f"Unexpected nvidia-smi query output: {query.stdout!r}")
gpu_row = dict(zip(query_fields, vals))

log(f"Complete nvidia-smi -q saved to: {q_path}")
log(f"GPU index: {gpu_row['index']}")
log(f"GPU name: {gpu_row['name']}")
log(f"GPU UUID: {gpu_row['uuid']}")
log(f"Driver version: {gpu_row['driver_version']}")
log(f"Driver-supported CUDA version: {driver_cuda_version}")
log(f"VRAM: {gpu_row['memory.total']} MiB")
log(f"Reported power limit: {gpu_row['power.limit']} W")

## Part A.3 — NVIDIA vendor specifications

The dictionary below contains the official values needed for this assignment for the two allowed cards.

For the percentage-of-theoretical calculation:
- FP32 uses peak non-Tensor FP32.
- TF32 uses dense Tensor TF32.
- FP16/BF16 use dense Tensor throughput with FP32 accumulation.
- FP8 is included only for the optional lower-precision probe.

Do **not** use the sparse number unless your benchmark explicitly uses structured sparsity.

In [ ]:
CARD_DATABASE = {
    "RTX 4090": {
        "architecture": "NVIDIA Ada Lovelace",
        "memory_type": "24 GB GDDR6X",
        "memory_bandwidth_GBs": 1008.0,
        "tensor_core_generation": "4th Generation",
        "tensor_supported_precisions": ["TF32", "FP16", "BF16", "FP8", "INT8", "INT4"],
        "theoretical_tflops_dense": {
            "FP32": 82.6,
            "TF32": 82.6,
            "FP16": 165.2,
            "BF16": 165.2,
            "FP8": 330.3,
        },
        "source": "NVIDIA Ada GPU Architecture whitepaper, Appendix A",
        "source_url": "https://images.nvidia.com/aem-dam/Solutions/geforce/ada/nvidia-ada-gpu-architecture.pdf",
        "source_accessed_utc": "2026-09-17",
        "supported_for_hw25": True,
    },
    "RTX 5090": {
        "architecture": "NVIDIA Blackwell",
        "memory_type": "32 GB GDDR7",
        "memory_bandwidth_GBs": 1792.0,
        "tensor_core_generation": "5th Generation",
        "tensor_supported_precisions": ["TF32", "FP16", "BF16", "FP8", "FP6", "FP4", "INT8"],
        "theoretical_tflops_dense": {
            "FP32": 104.8,
            "TF32": 104.8,
            "FP16": 209.5,
            "BF16": 209.5,
            "FP8": 419.0,
            "FP4": 1676.0,
        },
        "source": "NVIDIA RTX Blackwell GPU Architecture whitepaper, Appendix A",
        "source_url": "https://images.nvidia.com/aem-dam/Solutions/geforce/blackwell/nvidia-rtx-blackwell-gpu-architecture.pdf",
        "source_accessed_utc": "2026-09-17",
        "supported_for_hw25": True,
    },
}

detected_name = gpu_row["name"]
card_key = next((k for k in CARD_DATABASE if k in detected_name), None)

if card_key is None:
    raise RuntimeError(
        f"Detected {detected_name!r}. HW2.5 requires an RTX 4090 or RTX 5090."
    )

card_specs = CARD_DATABASE[card_key]

gpu_info = {
    "name": detected_name,
    "nvidia_smi_index": int(gpu_row["index"]),
    "uuid": gpu_row["uuid"],
    "driver_version": gpu_row["driver_version"],
    "driver_cuda_version": driver_cuda_version,
    "pytorch_version": torch.__version__,
    "pytorch_cuda_version": torch.version.cuda,
    "vram_mib": float(gpu_row["memory.total"]),
    "power_limit_w": float(gpu_row["power.limit"]),
    "hw25_supported_gpu": True,
    "card_specs": card_specs,
    "captured_at_utc": utc_now(),
    "nvidia_smi_q_file": str(q_path),
}

(ARTIFACT_DIR / "gpu_info.json").write_text(json.dumps(gpu_info, indent=2))

spec_df = pd.DataFrame([{
    "GPU": detected_name,
    "UUID": gpu_info["uuid"],
    "Architecture": card_specs["architecture"],
    "Memory": card_specs["memory_type"],
    "Bandwidth (GB/s)": card_specs["memory_bandwidth_GBs"],
    "Tensor Cores": card_specs["tensor_core_generation"],
    "Reduced precisions": ", ".join(card_specs["tensor_supported_precisions"]),
    "Driver CUDA": driver_cuda_version,
    "PyTorch CUDA build": torch.version.cuda,
    "Source": card_specs["source_url"],
    "Source accessed": card_specs["source_accessed_utc"],
}])
display(spec_df)

log("Vendor specification:")
for k, v in card_specs.items():
    log(f"  {k}: {v}")
log(f"GPU {detected_name} satisfies the HW2.5 hardware requirement.")


## Part A.1 — reservation and GPU-hour record

The notebook cannot invent your reservation history. It creates a template. Replace the bracketed fields with the actual reservation information and keep any reservation screenshot/export in the repository.

In [ ]:
reservation_path = ARTIFACT_DIR / "reservation_gpu_hours.md"

# Preserve a completed record. Create the template only when absent.
if not reservation_path.exists():
    reservation_path.write_text(f"""# HW2.5 GPU Reservation and Usage

- Student SID4: {SID4:04d}
- GPU: {gpu_info['name']}
- GPU UUID: `{gpu_info['uuid']}`
- Reservation date/time: [FILL IN]
- Reservation ID / evidence: [FILL IN]
- Reserved duration (hours): [FILL IN]
- Actual GPU hours consumed: [FILL IN]
- Notes: [FILL IN]
""")

log(f"Reservation record: {reservation_path}")
print("Fill every [FILL IN] field before final submission.")


## Environment snapshot and reproducibility record

This captures installed Python packages. Keep the executed notebook outputs intact in Git.

In [ ]:
freeze = run_cmd([sys.executable, "-m", "pip", "freeze"])
(ARTIFACT_DIR / "pip_freeze.txt").write_text(freeze.stdout)

env = {
    "timestamp_utc": utc_now(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda_build": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu": gpu_info
}
(ARTIFACT_DIR / "environment.json").write_text(json.dumps(env, indent=2, default=str))
log("Saved pip_freeze.txt and environment.json")

## Git checkpoint

The course standing requirements say incremental commits matter. After this notebook runs successfully, commit it and the generated provenance files, for example:

```bash
git add 00_hw2_5_setup_provenance.ipynb hw2_5_artifacts/
git commit -m "HW2.5 setup and GPU provenance"
```

Do not create the final `hw2-5` tag until all notebooks and outputs are complete.